# Moduł 2.4 — Riemannian Tangent Space + Logistic Regression (EEGMMIDB, LOSO)

**Cel:** zbudować klasyfikator EEG oparty na geometrii macierzy kowariancji, porównywalny z wcześniejszymi modułami 2 (CSP + LDA), 2.2 (FBCSP + LDA) i 2.3 (EEGNet-8,2). Notebook **nie pobiera EDF i nie przygotowuje danych na nowo**. Wykorzystuje *ten sam* zapis z modułu 1: `X.npy`, `y.npy`, `groups.npy`, `runs.npy`, `manifest.json` i `podzialy_loso.json`.

**Pipeline:** kowariancja epoki z regularizacją OAS → projekcja macierzy SPD do riemannowskiej przestrzeni stycznej (`TangentSpace`) → standaryzacja cech → regresja logistyczna. Wyznaczanie średniej referencyjnej przestrzeni stycznej, standaryzacji i parametrów klasyfikatora odbywa się **wyłącznie na treningowych uczestnikach** w danym foldzie LOSO. `tsupdate=False`: nie dopasowujemy średniej do testowych epok.

**Protokół:** identyczne `dataset_id`, foldy LOSO, klasy `1=T1`, `2=T2`, domyślny próg `0,50`, opcjonalne dobranie progu na osobach walidacyjnych, zapis plików `wyniki_per_subject.csv`, `predykcje_epok.csv`, `podzialy_uczestnikow.csv` i modelu `modele/Sxxx.joblib`. Na końcu wyświetlane są wyniki, wykresy, macierz pomyłek i wybrane predykcje.

**Źródła i granice reprodukcji:**

- Barachant A., Bonnet S., Congedo M., Jutten C. (2012), [*Multiclass Brain–Computer Interface Classification by Riemannian Geometry*](https://doi.org/10.1109/TBME.2011.2172210), *IEEE Transactions on Biomedical Engineering* 59(4), 920–928. Opisano m.in. rzutowanie kowariancji do przestrzeni stycznej i **TSLDA**.
- Congedo M., Barachant A., Bhatia R. (2017), [*Riemannian geometry for EEG-based brain-computer interfaces; a primer and a review*](https://doi.org/10.1080/2326263X.2017.1297192), *Brain-Computer Interfaces* 4(3), 155–174.
- [Dokumentacja pyRiemann — TangentSpace](https://pyriemann.readthedocs.io/en/latest/generated/pyriemann.tangentspace.TangentSpace.html) i [Covariances](https://pyriemann.readthedocs.io/en/latest/generated/pyriemann.estimation.Covariances.html).

**To nie jest identyczna reprodukcja TSLDA z Barachant (2012):** na życzenie klasyfikatorem jest **Logistic Regression**, nie LDA; kowariancje są regularizowane OAS, a dane wejściowe pochodzą ze wspólnego preprocessingu modułu 1 (domyślnie 8–30 Hz). Jest to wariant **TS + LR** reprezentujący rodzinę riemannowską.

**Uwaga o zadaniu:** przy `RUNS=[3,4,7,8,11,12]` w module 1 analizowane są jednocześnie *motor execution* i *motor imagery*; wyłącznie wyobrażanie lewej/prawej dłoni wymaga wygenerowania innego `dataset_id` z `RUNS=[4,8,12]` i użycia go we **wszystkich** modelach porównawczych.

**Zależności:** `pip install pyriemann scikit-learn numpy scipy pandas matplotlib joblib`. `pyriemann` musi być zainstalowany w kernelu Jupyter użytym do uruchomienia notebooka. Nie uruchamiaj `joblib.load()` na plikach z niezaufanego źródła.


In [4]:
from pathlib import Path
import hashlib
import json
import os
import platform
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import stats
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

try:
    import pyriemann
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
except ImportError as exc:
    raise ImportError(
        'Brak biblioteki pyriemann. W kernelu notebooka wykonaj: %pip install pyriemann, '
        'a następnie uruchom kernel ponownie.'
    ) from exc

# ===== WSPÓLNE DANE Z MODUŁU 1 =====
DATA_BASE = Path.cwd() / 'dane_eegmmidb'
MANIFEST_PATH = None  # None = ostatni zestaw przygotowany przez moduł 1
RESULTS_BASE = Path.cwd() / 'wyniki_modeli' / 'RIEMANN_TS_LR'
VERIFY_FILE_HASHES = True

# ===== MODEL =====
COV_ESTIMATOR = 'oas'   # OAS: kowariancja dodatnio określona przy referencji average
TANGENT_METRIC = 'riemann'  # średnia i mapa logarytmiczna AIRM
LR_C = 1.0            # regularizacja L2; wartość ustalona przed testem
LR_MAX_ITER = 2000
LR_CLASS_WEIGHT = None  # bez dostrajania na danych testowych
SEED = 42

# ===== TEN SAM PROTOKÓŁ WALIDACJI I PROGU CO W CSP + LDA =====
TUNE_THRESHOLD = False
DEFAULT_THRESHOLD = 0.50
MIN_VALIDATION_RECALL = 0.20
MIN_VALIDATION_POSITIVES = 3
REFIT_ON_ALL_NON_TEST = True
SAVE_MODELS = True

# Do prób diagnostycznych można użyć np. MAX_FOLDS = 4.
# Pełne, porównywalne wyniki LOSO wymagają MAX_FOLDS = None.
MAX_FOLDS = None

# ===== PREZENTACJA WYNIKÓW =====
SHOW_PLOTS = True
SAVE_PLOTS = True
TEST_SUBJECT_TO_SHOW = None
N_SAMPLES = 10
VERIFY_SAVED_MODEL = True

assert LR_C > 0 and LR_MAX_ITER > 0
assert 0 < DEFAULT_THRESHOLD < 1
assert 0 <= MIN_VALIDATION_RECALL <= 1
assert MIN_VALIDATION_POSITIVES >= 1 and N_SAMPLES >= 1
assert MAX_FOLDS is None or (isinstance(MAX_FOLDS, int) and MAX_FOLDS >= 1)

print('Python:', platform.python_version(), '| sklearn:', sklearn.__version__,
      '| pyriemann:', pyriemann.__version__, '| NumPy:', np.__version__)


ImportError: Brak biblioteki pyriemann. W kernelu notebooka wykonaj: %pip install pyriemann, a następnie uruchom kernel ponownie.

## 1. Wczytanie zestawu i kontrola identycznych podziałów LOSO

Odczytujemy dane tylko raz z modułu 1 (tablica `X` jest mapowana z dysku). Weryfikujemy SHA-256, rozmiary i typy danych, identyfikatory osób, klasy i nieprzecinające się zbiory treningowe/walidacyjne/testowe. Dane EEG pozostają w **woltach**; nie dokonujemy dodatkowej normalizacji na podstawie zbioru testowego.


In [ ]:
def read_json(path):
    with Path(path).open(encoding='utf-8') as f:
        return json.load(f)


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(block_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(payload, size=12):
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()[:size]


if MANIFEST_PATH is None:
    pointer = read_json(DATA_BASE / 'ostatni_zestaw.json')
    manifest_path = (DATA_BASE / pointer['manifest_relative_path']).resolve()
else:
    manifest_path = Path(MANIFEST_PATH).expanduser().resolve()

manifest = read_json(manifest_path)
if manifest.get('schema_version') != '1.0':
    raise ValueError('Nieobsługiwana wersja schematu modułu 1.')
dataset_dir = manifest_path.parent
for key, info in manifest['files'].items():
    file_path = dataset_dir / info['path']
    if not file_path.is_file():
        raise FileNotFoundError(file_path)
    if VERIFY_FILE_HASHES and sha256_file(file_path) != info['sha256']:
        raise ValueError(f'Niezgodny SHA-256 — {file_path}')

X_all = np.load(dataset_dir / manifest['files']['X']['path'], mmap_mode='r')
y_all = np.load(dataset_dir / manifest['files']['y']['path'], mmap_mode='r')
groups = np.load(dataset_dir / manifest['files']['groups']['path'], mmap_mode='r')
epoch_run = np.load(dataset_dir / manifest['files']['runs']['path'], mmap_mode='r')
splits = read_json(dataset_dir / manifest['files']['splits']['path'])
SUBJECT_IDS = [int(s) for s in manifest['subject_ids']]
CLASS_NAMES = {int(k): v for k, v in manifest['labels'].items()}
POSITIVE_LABEL = int(manifest['positive_label'])

if X_all.dtype != np.float32 or X_all.ndim != 3 or list(X_all.shape) != manifest['X_shape']:
    raise ValueError('Niezgodny format X; wymagane float32 i [epoki, kanały, próbki].')
if not (len(X_all) == len(y_all) == len(groups) == len(epoch_run) == manifest['n_epochs']):
    raise ValueError('Niezgodne liczby epok między plikami.')
if y_all.dtype != np.int8 or groups.dtype != np.int16 or epoch_run.dtype != np.int8:
    raise ValueError('Niezgodne typy y/groups/runs.')
if np.unique(groups).tolist() != SUBJECT_IDS or set(np.unique(y_all)) != {1, 2}:
    raise ValueError('Niepoprawne etykiety klas lub uczestników.')
if POSITIVE_LABEL != 2:
    raise ValueError('Wymagane etykiety: T1=1 i T2=2.')
if X_all.shape[1] < 2 or X_all.shape[2] < 4:
    raise ValueError('Zbyt mało kanałów lub próbek do obliczenia kowariancji.')
if splits['dataset_id'] != manifest['dataset_id']:
    raise ValueError('Podziały nie należą do wybranego zestawu danych.')
if [int(f['test_subject']) for f in splits['folds']] != SUBJECT_IDS:
    raise ValueError('Podziały LOSO nie obejmują każdej osoby dokładnie raz.')
for f in splits['folds']:
    train, val, test = (set(f[key]) for key in (
        'train_subjects_internal', 'validation_subjects', 'test_subjects'))
    if train & val or train & test or val & test or train | val | test != set(SUBJECT_IDS):
        raise ValueError(f'Niepoprawny podział train/val/test w foldzie {f["fold"]}.')
    if test != {int(f['test_subject'])}:
        raise ValueError('Niepoprawny test_subject w foldzie LOSO.')

folds_to_run = splits['folds'] if MAX_FOLDS is None else splits['folds'][:MAX_FOLDS]
if not folds_to_run:
    raise ValueError('Brak foldów do przetworzenia.')

print('dataset_id:', manifest['dataset_id'], '| X:', X_all.shape,
      '| liczba osób:', len(SUBJECT_IDS))
print('Klasy:', CLASS_NAMES, '| zadanie:', manifest['task_types'])
print('Wykonanie LOSO:', len(folds_to_run), '/', len(SUBJECT_IDS), 'foldów')
print('Liczba cech tangent space:', X_all.shape[1] * (X_all.shape[1] + 1) // 2)

# Każda epoka jest transformowana niezależnie; OAS stabilizuje macierz SPD.
example_covs = Covariances(estimator=COV_ESTIMATOR).transform(
    np.asarray(X_all[:1], dtype=np.float64)
)
if np.linalg.eigvalsh(example_covs[0]).min() <= 0:
    raise ValueError('Przykładowa kowariancja nie jest SPD. Zmień sposób estymacji.')
print('Test SPD pierwszej epoki: poprawny')


## 2. Model: kowariancje → riemannowska przestrzeń styczna → Logistic Regression

Dla każdej epoki obliczamy regularizowaną macierz kowariancji SPD o wymiarach `[kanały × kanały]`. W danym foldzie wyznaczamy **średnią riemannowską tylko z macierzy treningowych** i rzutujemy macierze kowariancji do przestrzeni stycznej. Przestrzeń styczna daje wektor `n_kanałów × (n_kanałów+1) / 2` cech (górny trójkąt z właściwym skalowaniem poza przekątną). `StandardScaler` i regresja logistyczna są kolejnymi etapami pipeline'u. Wszystkie obiekty są standardowymi klasami bibliotek i zapis `.joblib` można wczytać w module 3.

**Ważne:** `TangentSpace(metric='riemann', tsupdate=False)` nie przelicza referencji na próbach walidacyjnych ani testowych. Dodatnia określoność przy referencji średniej EEG jest wspierana przez estymację OAS. Dobór `LR_C` i estymatora kowariancji przed analizą testową zabezpiecza przed dostrajaniem po wynikach LOSO.


In [ ]:
def make_model():
    return Pipeline([
        ('ToFloat64', FunctionTransformer(
            np.asarray, validate=False, kw_args={'dtype': np.float64},
        )),
        ('Covariances', Covariances(estimator=COV_ESTIMATOR)),
        ('TangentSpace', TangentSpace(metric=TANGENT_METRIC, tsupdate=False)),
        ('Scaler', StandardScaler()),
        ('LR', LogisticRegression(
            C=LR_C, penalty='l2', solver='lbfgs', max_iter=LR_MAX_ITER,
            random_state=SEED, class_weight=LR_CLASS_WEIGHT,
        )),
    ])


def positive_probabilities(model, X):
    index = np.flatnonzero(model.classes_ == POSITIVE_LABEL)
    if len(index) != 1:
        raise ValueError('Wytrenowany model nie zawiera klasy T2.')
    return model.predict_proba(X)[:, int(index[0])]


def labels_from_probability(prob, threshold):
    return np.where(prob >= threshold, POSITIVE_LABEL, 1).astype(np.int8)


def choose_validation_threshold(y_true, prob):
    def scores(threshold):
        pred = labels_from_probability(prob, threshold)
        return (float(precision_score(y_true, pred, pos_label=2, zero_division=0)),
                float(recall_score(y_true, pred, pos_label=2, zero_division=0)))

    if not TUNE_THRESHOLD:
        p, r = scores(DEFAULT_THRESHOLD)
        return float(DEFAULT_THRESHOLD), 'fixed', p, r
    candidates = []
    for threshold in np.linspace(0.05, 0.95, 91):
        pred = labels_from_probability(prob, threshold)
        p, r = scores(threshold)
        if r >= MIN_VALIDATION_RECALL and int((pred == 2).sum()) >= MIN_VALIDATION_POSITIVES:
            candidates.append((p, r, -abs(threshold - 0.5), float(threshold)))
    if not candidates:
        p, r = scores(DEFAULT_THRESHOLD)
        return float(DEFAULT_THRESHOLD), 'fallback_0.5', p, r
    p, r, _, threshold = max(candidates)
    return float(threshold), 'validation_optimized', p, r


def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[1, 2]).ravel()
    return {
        'precision_T2': float(precision_score(y_true, y_pred, pos_label=2, zero_division=0)),
        'recall_T2': float(recall_score(y_true, y_pred, pos_label=2, zero_division=0)),
        'f1_T2': float(f1_score(y_true, y_pred, pos_label=2, zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'n_test_epochs': int(len(y_true)),
        'n_predicted_T2': int((y_pred == 2).sum()),
    }


## 3. Identyfikator eksperymentu i katalog wyników

Konfiguracja modelu trafia do `manifest_modelu.json`, a wynikowy katalog ma strukturę `wyniki_modeli/RIEMANN_TS_LR/<dataset_id>/<experiment_id>/`. Dwie konfiguracje modelu nie nadpiszą swoich wyników. W plikach CSV zachowujemy identyczne kolumny co w CSP + LDA i FBCSP + LDA; dotyczy to również raportowania predykcji dla klasy dodatniej T2.


In [ ]:
TRAIN_CONFIG = {
    'model': 'RIEMANN_TS_LR',
    'dataset_id': manifest['dataset_id'],
    'covariance_estimator': COV_ESTIMATOR,
    'tangent_space_metric': TANGENT_METRIC,
    'tangent_space_tsupdate': False,
    'n_channels': int(X_all.shape[1]),
    'n_tangent_features': int(X_all.shape[1] * (X_all.shape[1] + 1) // 2),
    'scaler': 'StandardScaler',
    'classifier': 'LogisticRegression',
    'LR_C': LR_C, 'LR_penalty': 'l2', 'LR_solver': 'lbfgs',
    'LR_max_iter': LR_MAX_ITER, 'LR_class_weight': LR_CLASS_WEIGHT,
    'seed': SEED,
    'tune_threshold': TUNE_THRESHOLD,
    'default_threshold': DEFAULT_THRESHOLD,
    'min_validation_recall': MIN_VALIDATION_RECALL,
    'min_validation_positives': MIN_VALIDATION_POSITIVES,
    'refit_on_all_non_test': REFIT_ON_ALL_NON_TEST,
    'max_folds': MAX_FOLDS,
    'positive_label': POSITIVE_LABEL,
    'save_models': SAVE_MODELS,
    'versions': {
        'python': platform.python_version(), 'numpy': np.__version__,
        'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
        'pyriemann': pyriemann.__version__,
    },
}
TRAIN_HASH = stable_hash(TRAIN_CONFIG)
OUTPUT_DIR = RESULTS_BASE / manifest['dataset_id'] / TRAIN_HASH
MODEL_DIR = OUTPUT_DIR / 'modele'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_MODELS:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

experiment_manifest = {
    'experiment_id': TRAIN_HASH, 'model': 'RIEMANN_TS_LR',
    'dataset_id': manifest['dataset_id'],
    'data_manifest_relative_path': os.path.relpath(manifest_path, OUTPUT_DIR),
    'config': TRAIN_CONFIG,
    'n_folds_expected': len(SUBJECT_IDS),
    'n_folds_planned': len(folds_to_run),
    'status': 'running',
    'model_file_pattern': 'modele/S{subject:03d}.joblib' if SAVE_MODELS else None,
    'results': {
        'per_subject': 'wyniki_per_subject.csv',
        'predictions': 'predykcje_epok.csv',
        'assignments': 'podzialy_uczestnikow.csv',
    },
}
with (OUTPUT_DIR / 'manifest_modelu.json').open('w', encoding='utf-8') as f:
    json.dump(experiment_manifest, f, indent=2, ensure_ascii=False)
print('Folder eksperymentu:', OUTPUT_DIR)
print('Modele na dysku:', SAVE_MODELS)


## 4. Pętla LOSO: identyczne osoby treningowe, walidacyjne i testowe

Każda osoba jest zewnętrznym testem dokładnie raz. `train_subjects_internal` służą do dopasowania pierwszego modelu i przestrzeni stycznej; `validation_subjects` służą **wyłącznie** do sprawdzenia/wyboru progu. Przy `REFIT_ON_ALL_NON_TEST=True` dopasowujemy od nowa kompletny pipeline na wszystkich osobach poza testową (jak w CSP + LDA). Dopiero potem przewidujemy etykiety epok osoby testowej. Również w drugim dopasowaniu przestrzeń styczna jest wyznaczana bez danych testowych.

Po każdym foldzie zapisywane są wyniki na dysk. **Nie dobieraj parametrów modelu na podstawie metryk testowych**: poprawną metodą strojenia `C` byłaby dodatkowa walidacja wewnętrzna, a nie wybór najlepszego wyniku zewnętrznego LOSO.


In [ ]:
fold_results, fold_assignments, predictions = [], [], []
for fold in folds_to_run:
    start_time = time.perf_counter()
    fold_no = int(fold['fold'])
    test_subject = int(fold['test_subject'])
    train_people = fold['train_subjects_internal']
    val_people = fold['validation_subjects']

    train_idx = np.flatnonzero(np.isin(groups, train_people))
    val_idx = np.flatnonzero(np.isin(groups, val_people))
    test_idx = np.flatnonzero(groups == test_subject)
    non_test_idx = np.flatnonzero(groups != test_subject)
    if len(train_idx) + len(val_idx) + len(test_idx) != len(groups):
        raise AssertionError(f'Fold {fold_no}: niekompletny podział epok.')
    if any(len(ix) == 0 or set(np.unique(y_all[ix])) != {1, 2}
           for ix in (train_idx, val_idx, test_idx)):
        raise ValueError(f'Fold {fold_no}: brak epok lub obu klas w podziale.')

    validation_model = make_model()
    validation_model.fit(X_all[train_idx], y_all[train_idx])
    validation_prob = positive_probabilities(validation_model, X_all[val_idx])
    threshold, method, val_precision, val_recall = choose_validation_threshold(
        y_all[val_idx], validation_prob,
    )

    if REFIT_ON_ALL_NON_TEST:
        final_model = make_model()
        final_model.fit(X_all[non_test_idx], y_all[non_test_idx])
    else:
        final_model = validation_model

    test_prob = positive_probabilities(final_model, X_all[test_idx])
    test_pred = labels_from_probability(test_prob, threshold)
    metrics = calculate_metrics(y_all[test_idx], test_pred)

    if SAVE_MODELS:
        joblib.dump(final_model, MODEL_DIR / f'S{test_subject:03d}.joblib', compress=3)

    duration = round(time.perf_counter() - start_time, 3)
    fold_results.append({
        'fold': fold_no, 'test_subject': test_subject,
        'dataset_id': manifest['dataset_id'],
        'n_train_subjects_internal': len(train_people),
        'n_validation_subjects': len(val_people),
        'n_train_subjects_final': (len(SUBJECT_IDS) - 1 if REFIT_ON_ALL_NON_TEST
                                   else len(train_people)),
        'n_train_epochs_internal': int(len(train_idx)),
        'n_validation_epochs': int(len(val_idx)),
        'threshold_T2': threshold, 'threshold_method': method,
        'validation_precision_T2': val_precision,
        'validation_recall_T2': val_recall,
        'training_seconds': duration,
        **metrics,
    })
    fold_assignments.append({
        'fold': fold_no, 'test_subject': test_subject,
        'train_subjects_internal': ','.join(map(str, train_people)),
        'validation_subjects': ','.join(map(str, val_people)),
        'test_subjects': str(test_subject),
        'final_train_subjects': ','.join(map(str,
            [s for s in SUBJECT_IDS if s != test_subject]
            if REFIT_ON_ALL_NON_TEST else train_people)),
    })
    for test_epoch, (global_index, pred, prob) in enumerate(
        zip(test_idx, test_pred, test_prob), start=1,
    ):
        predictions.append({
            'fold': fold_no, 'test_subject': test_subject,
            'test_epoch': test_epoch, 'global_epoch_index': int(global_index),
            'run': int(epoch_run[global_index]),
            'y_true': int(y_all[global_index]), 'y_pred': int(pred),
            'probability_T2': float(prob), 'threshold_T2': float(threshold),
        })

    pd.DataFrame(fold_results).to_csv(OUTPUT_DIR / 'wyniki_per_subject.csv', index=False)
    pd.DataFrame(fold_assignments).to_csv(
        OUTPUT_DIR / 'podzialy_uczestnikow.csv', index=False,
    )
    pd.DataFrame(predictions).to_csv(OUTPUT_DIR / 'predykcje_epok.csv', index=False)

    print(f'[{len(fold_results):03d}/{len(folds_to_run):03d}] test S{test_subject:03d} '
          f'| precision T2={metrics["precision_T2"]:.3f} '
          f'| recall T2={metrics["recall_T2"]:.3f} '
          f'| balanced accuracy={metrics["balanced_accuracy"]:.3f} '
          f'| {duration:.1f} s')

results_df = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(predictions)
assert results_df['test_subject'].tolist() == [int(f['test_subject']) for f in folds_to_run]
expected_epochs = int(np.isin(groups, results_df['test_subject']).sum())
assert len(predictions_df) == expected_epochs
assert not predictions_df['global_epoch_index'].duplicated().any()

is_complete_loso = len(folds_to_run) == len(SUBJECT_IDS)
if is_complete_loso:
    assert len(predictions_df) == len(X_all)
    assert set(predictions_df['global_epoch_index']) == set(range(len(X_all)))

experiment_manifest['status'] = 'complete' if is_complete_loso else 'partial'
experiment_manifest['n_folds_completed'] = len(fold_results)
with (OUTPUT_DIR / 'manifest_modelu.json').open('w', encoding='utf-8') as f:
    json.dump(experiment_manifest, f, indent=2, ensure_ascii=False)

# Wskaźnik ostatniego pełnego eksperymentu — nie nadpisujemy go diagnostycznym LOSO.
if is_complete_loso:
    RESULTS_BASE.mkdir(parents=True, exist_ok=True)
    with (RESULTS_BASE / 'ostatni_eksperyment.json').open('w', encoding='utf-8') as f:
        json.dump({
            'experiment_relative_path': str(OUTPUT_DIR.relative_to(RESULTS_BASE)),
            'dataset_id': manifest['dataset_id'],
            'experiment_id': TRAIN_HASH,
        }, f, indent=2)

print('Zakończono:', len(results_df), 'foldów,', len(predictions_df), 'predykcji.')
print('Status:', experiment_manifest['status'], '| Pliki:', OUTPUT_DIR)


## 5. Wyniki treningu: tabela per osoba, przedziały ufności i wykresy

Każda osoba ma **równą wagę** w średniej i przedziale ufności (95% CI t-Studenta). Metryki *precision*, *recall* i *F1* dotyczą T2. Gdy `MAX_FOLDS` nie jest `None`, wyniki mają charakter **diagnostyczny** i nie stanowią pełnej oceny LOSO.


In [ ]:
METRIC_COLS = ["precision_T2", "recall_T2", "f1_T2", "balanced_accuracy", "accuracy"]
summary_rows = []
for metric in METRIC_COLS:
    values = results_df[metric].to_numpy(dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    sd = float(np.std(values, ddof=1)) if n > 1 else 0.0
    if n <= 1 or np.isclose(sd, 0.0):
        ci_low = ci_high = mean
    else:
        ci_low, ci_high = stats.t.interval(
            0.95, df=n - 1, loc=mean, scale=stats.sem(values)
        )
    summary_rows.append({
        "metric": metric, "n_subjects": n,
        "mean": mean, "std": sd,
        "CI95_lower": float(ci_low), "CI95_upper": float(ci_high),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "podsumowanie_loso.csv", index=False)

print("Wyniki dla wszystkich uczestników testowych:")
display(results_df[[
    "test_subject", "precision_T2", "recall_T2", "f1_T2",
    "balanced_accuracy", "accuracy", "n_test_epochs",
    "n_predicted_T2", "threshold_T2",
]].round(3))
print("Podsumowanie LOSO (każda osoba ma równą wagę):")
display(summary_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for metric, label in [
    ("precision_T2", "Precision T2"),
    ("recall_T2", "Recall T2"),
    ("balanced_accuracy", "Balanced accuracy"),
]:
    axes[0].plot(results_df["test_subject"], results_df[metric],
                 marker="o", markersize=3, linewidth=1.0, label=label)
axes[0].set(title="Wyniki LOSO dla każdej osoby", xlabel="Osoba testowa",
            ylabel="Wartość metryki", ylim=(0, 1.05))
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize="small")

axes[1].boxplot([results_df[metric].to_numpy() for metric in METRIC_COLS])
axes[1].set_xticks(range(1, len(METRIC_COLS) + 1))
axes[1].set_xticklabels(METRIC_COLS, rotation=35, ha="right")
axes[1].set(title="Rozkład metryk między uczestnikami", ylabel="Wartość metryki",
            ylim=(0, 1.05))
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(OUTPUT_DIR / "wyniki_loso.png", dpi=160, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


## 6. Macierz pomyłek i raport klasyfikacji

Poniższa macierz agreguje epoki testowe ze wszystkich ukończonych foldów (**ważenie po epokach**, nie po osobach). Podstawą porównania algorytmów pozostają metryki liczone **na uczestnikach**. Raport i wykres są zapisywane razem z CSV.


In [ ]:
y_true_all = predictions_df["y_true"].to_numpy(dtype=np.int8)
y_pred_all = predictions_df["y_pred"].to_numpy(dtype=np.int8)
report_text = classification_report(
    y_true_all, y_pred_all, labels=[1, 2],
    target_names=[CLASS_NAMES[1], CLASS_NAMES[2]], zero_division=0,
)
print("Raport klasyfikacji (agregacja po epokach):")
print(report_text)
(OUTPUT_DIR / "raport_klasyfikacji.txt").write_text(report_text, encoding="utf-8")

cm = confusion_matrix(y_true_all, y_pred_all, labels=[1, 2])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[CLASS_NAMES[1], CLASS_NAMES[2]],
).plot(values_format="d", cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"Tangent Space + LR: macierz pomyłek ({len(results_df)} osób)")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(OUTPUT_DIR / "macierz_pomylek.png", dpi=160, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


## 7. Przykładowe predykcje i odtworzenie modelu po zapisaniu

Wybierz uczestnika testowego i sprawdź przykładowe klasy i prawdopodobieństwa. Wczytanie `.joblib` odtwarza cały pipeline: regularizowaną kowariancję, referencję riemannowską, standaryzację oraz regresję. Weryfikujemy zgodność przewidywanych prawdopodobieństw z plikiem CSV dla **wszystkich epok jednej osoby** (zapisanych wcześniej poza treningiem).


In [ ]:
subject_to_show = (int(results_df['test_subject'].iloc[-1]) if TEST_SUBJECT_TO_SHOW is None
                   else int(TEST_SUBJECT_TO_SHOW))
if subject_to_show not in set(results_df['test_subject']):
    raise ValueError(f'S{subject_to_show:03d} nie jest w uruchomionych foldach.')
chosen = (predictions_df[predictions_df['test_subject'] == subject_to_show]
          .sort_values('test_epoch').reset_index(drop=True))
print(f'Przykładowe predykcje S{subject_to_show:03d}: '
      f'{min(N_SAMPLES, len(chosen))} z {len(chosen)} epok')
examples = chosen[[
    'test_epoch', 'run', 'y_true', 'y_pred', 'probability_T2', 'threshold_T2',
]].head(N_SAMPLES).copy()
examples['prawda'] = examples['y_true'].map(CLASS_NAMES)
examples['predykcja'] = examples['y_pred'].map(CLASS_NAMES)
examples['poprawnie'] = examples['y_true'].eq(examples['y_pred'])
display(examples[[
    'test_epoch', 'run', 'prawda', 'predykcja',
    'probability_T2', 'threshold_T2', 'poprawnie',
]].round(3))

if VERIFY_SAVED_MODEL and SAVE_MODELS:
    model_path = MODEL_DIR / f'S{subject_to_show:03d}.joblib'
    loaded_model = joblib.load(model_path)  # tylko własne, zaufane modele
    ids = chosen['global_epoch_index'].to_numpy(dtype=np.int64)
    probabilities_again = positive_probabilities(loaded_model, X_all[ids])
    predicted_again = labels_from_probability(
        probabilities_again, chosen['threshold_T2'].to_numpy(dtype=float),
    )
    if not np.allclose(probabilities_again,
                       chosen['probability_T2'].to_numpy(dtype=float),
                       rtol=1e-6, atol=1e-8):
        raise AssertionError('Model po wczytaniu nie odtwarza P(T2) z CSV.')
    if not np.array_equal(predicted_again, chosen['y_pred'].to_numpy()):
        raise AssertionError('Model po wczytaniu nie odtwarza klas z CSV.')
    ref = loaded_model.named_steps['TangentSpace'].reference_
    eig_min = float(np.min(np.linalg.eigvalsh(ref)))
    print(f'✓ Zgodność modelu po wczytaniu: {len(chosen)} epok; '
          f'min. wartość własna referencji = {eig_min:.3g}')
    print('Etapy:', list(loaded_model.named_steps))
elif VERIFY_SAVED_MODEL:
    print('SAVE_MODELS=False: pominięto odtwarzanie predykcji po wczytaniu.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(predictions_df.loc[predictions_df.y_true == 1, 'probability_T2'],
        bins=15, alpha=0.65, label=CLASS_NAMES[1])
ax.hist(predictions_df.loc[predictions_df.y_true == 2, 'probability_T2'],
        bins=15, alpha=0.65, label=CLASS_NAMES[2])
ax.axvline(DEFAULT_THRESHOLD, ls='--', lw=1.2, label='Próg domyślny 0.50')
ax.set(xlabel='Przewidziane P(T2)', ylabel='Liczba epok',
       title='Rozkład prawdopodobieństwa T2 na danych testowych LOSO')
ax.legend()
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(OUTPUT_DIR / 'rozklad_prawdopodobienstw_T2.png',
                dpi=160, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

print('Zapisane wyniki:', OUTPUT_DIR)
print('- manifest_modelu.json, wyniki_per_subject.csv, predykcje_epok.csv')
print('- podzialy_uczestnikow.csv, podsumowanie_loso.csv, raport_klasyfikacji.txt')
if SAVE_PLOTS:
    print('- wyniki_loso.png, macierz_pomylek.png, rozklad_prawdopodobienstw_T2.png')
if SAVE_MODELS:
    print('- modele/Sxxx.joblib — osobny pipeline dla każdej osoby testowej')


## 8. Porównanie z pozostałymi metodami i wykorzystanie modułu 3

1. **Użyj tego samego `dataset_id`** co w CSP + LDA, FBCSP + LDA i EEGNet-8,2. Pełne porównanie wymaga `MAX_FOLDS=None` oraz tych samych osób testowych.
2. **Porównaj `wyniki_per_subject.csv`** ze wszystkich modeli po `dataset_id` i `test_subject` (zwłaszcza *balanced accuracy*, F1 T2, precision T2 i recall T2). Nie porównuj wyłącznie macierzy pomyłek zagregowanej po epokach. Wyniki są sparowane po osobach.
3. **Moduł 3 (prezentacja bez ponownego treningu):** ustaw `RESULTS_BASE = Path.cwd() / 'wyniki_modeli' / 'RIEMANN_TS_LR'` oraz `SHOW_CSP_PATTERNS=False`. `VERIFY_MODEL_PREDICTIONS=True` działa dla zapisanego standardowego pipeline'u `.joblib` (potrzebny zainstalowany `pyriemann`).
4. **Interpretacja metodologiczna:** w odróżnieniu od CSP model nie uczy filtrów przestrzennych maksymalizujących wariancję klas. Rozróżnia klasy na podstawie geometrii **całych macierzy kowariancji** w przestrzeni stycznej. `TS + LogisticRegression` jest wariantem wobec opublikowanego `TSLDA` z Barachant (2012); nie przedstawiaj ich jako tej samej implementacji.
5. **Uwaga dotycząca czasu i pamięci:** dla 64 kanałów jedna epoka ma 2080 cech w przestrzeni stycznej. Pełne 109-fold LOSO może być obliczeniowo kosztowne; najpierw sprawdź `TEST_20` z modułu 1 albo diagnostyczne `MAX_FOLDS=4`, później uruchom pełne foldy. Nie zmieniaj `dataset_id` między modelami w publikowanym porównaniu.
